# 08 — Hybrid system & final evaluation
**Project:** Clinical Medication Extraction | **Phase 5c → deployment prep**

## The decision this notebook makes

You now have three detection approaches, each with measured, documented failure modes:

| Approach | Strength | Weakness |
|---|---|---|
| **Rules** | precise, fast (ms), 0% hallucination by construction, auditable | lexicon-bound; misses unknown drugs; crude negation scope |
| **Transformer** | finds drugs outside the lexicon | no normalization or status; extra false positives |
| **LLM** | handles narrative syntax, negation scope, multi-drug attribution | slow, can hallucinate, needs parsing scaffolding |

**The obvious move is to pick the winner. The better move is to notice they fail on different inputs.** Rules dominate on structured medication lists — `Lipitor 80 mg q.d` is a solved problem, and spending 4 seconds of GPU on it is waste. The LLM earns its cost on narrative prose where syntax carries the meaning.

So the hybrid is not a compromise. It's **routing**: send each input to the component that handles it best, and measure whether the routing actually pays.

That framing — from "which model wins" to "what should the system look like" — is the shift from model comparison to architecture, and it's what the ADR at the end records.

## Setup

In [1]:
import pandas as pd
import numpy as np
import json, os, sys, time
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''
WORK, SRC, GOLD, DOCS = BASE+'working/', BASE+'src/', BASE+'gold/', BASE+'docs/'
os.makedirs(DOCS, exist_ok=True)
sys.path.insert(0, SRC)

from sectionizer import split_sections
from evaluation import evaluate, match

work = pd.read_parquet(WORK + 'notes_subset.parquet')

def load(name):
    p = WORK + name
    return pd.read_parquet(p) if os.path.exists(p) else None

ex_rules = load('extractions_rules.parquet')
ex_ner   = load('extractions_ner.parquet')
ex_llm   = load('extractions_llm.parquet')

for n, d in [('rules', ex_rules), ('transformer', ex_ner), ('llm', ex_llm)]:
    print(f'{n:12} {len(d) if d is not None else "missing":>8}')

Mounted at /content/drive
rules            1540
transformer      1758
llm               271


---
# Part 1 — The routing rule

## Which sections go where

**Structured sections** (`medications`, `discharge_medications`, `allergies`) are lists. Rules handle these at ~99% of the LLM's quality for ~0.1% of the cost — and with a hallucination rate of exactly zero, which matters when the output is a medication list.

**Narrative sections** (`hpi`, `hospital_course`, `plan`, `assessment`) are prose. This is where negation scope, temporal context, and multi-drug attribution live — the three things rules demonstrably fail at.

**Why route on section rather than on confidence?** Because the sectionizer already gives you the signal for free, it's deterministic and auditable (a privacy officer can read the routing rule), and it maps onto the actual linguistic difference between list text and prose. Confidence-based routing would need calibration you haven't done and can't easily explain.

In [2]:
STRUCTURED_SECTIONS = {'medications', 'discharge_medications', 'allergies'}
NARRATIVE_SECTIONS = {'hpi', 'hospital_course', 'plan', 'assessment', 'assessment_plan',
                      'pmh', 'discharge_instructions', '_unsectioned'}

def route_sections(note):
    """Split a note's sections into the two processing lanes."""
    secs = split_sections(note)
    structured = {k: v for k, v in secs.items() if k in STRUCTURED_SECTIONS}
    narrative = {k: v for k, v in secs.items()
                 if k in NARRATIVE_SECTIONS or (k not in STRUCTURED_SECTIONS
                                                and not k.startswith('exam:'))}
    return structured, narrative

sample = work['transcription'].iloc[0]
s, n = route_sections(sample)
print('structured lane:', list(s.keys()))
print('narrative lane :', list(n.keys()))
print()
tot = sum(len(v) for v in {**s, **n}.values()) or 1
print(f'characters — structured {sum(len(v) for v in s.values())/tot:.0%}, '
      f'narrative {sum(len(v) for v in n.values())/tot:.0%}')

structured lane: ['allergies', 'medications']
narrative lane : ['hpi', 'pmh', 'family_history', 'social_history', 'physical_exam', 'labs', 'hospital_course']

characters — structured 6%, narrative 94%


In [3]:
# How much of the corpus would go to the expensive lane?
share = []
for t in work['transcription']:
    s, n = route_sections(t)
    sc, nc = sum(len(v) for v in s.values()), sum(len(v) for v in n.values())
    if sc + nc:
        share.append(nc / (sc + nc))
print(f'Median share of text routed to the LLM: {np.median(share):.0%}')
print(f'-> LLM cost is roughly {np.median(share):.0%} of the all-LLM baseline')

Median share of text routed to the LLM: 99%
-> LLM cost is roughly 99% of the all-LLM baseline


---
# Part 2 — Merging predictions

Two components produce predictions; the system emits one list. Rules for structured sections, LLM for narrative, deduplicated on `(note, drug, status)` with **rules winning ties** — because when both find the same event, the rules version carries verified dose attribution and cannot be hallucinated.

Every row keeps a `source` column. **Provenance is not optional in a clinical system**: when a clinician queries an output, "which component produced this and why" must be answerable. It's also what lets you compute per-component contribution below.

In [4]:
def merge_predictions(structured_preds, narrative_preds, prefer='rules'):
    a = structured_preds.copy(); a['source'] = 'rules'
    b = narrative_preds.copy();  b['source'] = 'llm'
    first, second = (a, b) if prefer == 'rules' else (b, a)
    merged = pd.concat([first, second], ignore_index=True)
    merged['_k'] = (merged['note_id'].astype(str) + '|'
                    + merged['normalized'].astype(str).str.lower().str.strip() + '|'
                    + merged['status'].astype(str).str.lower().str.strip())
    merged = merged.drop_duplicates('_k', keep='first').drop(columns='_k')
    return merged.reset_index(drop=True)

# verify on a hand-checkable case
_s = pd.DataFrame([{'note_id':1,'normalized':'atorvastatin','status':'active','dose':'80 mg'},
                   {'note_id':1,'normalized':'warfarin','status':'active','dose':None}])
_n = pd.DataFrame([{'note_id':1,'normalized':'atorvastatin','status':'active','dose':None},
                   {'note_id':1,'normalized':'ramelteon','status':'active','dose':'8 mg'}])
print(merge_predictions(_s, _n).to_string(index=False))
print()
print('atorvastatin kept the rules row (with its dose); ramelteon came only from the LLM.')

 note_id   normalized status  dose source
       1 atorvastatin active 80 mg  rules
       1     warfarin active  None  rules
       1    ramelteon active  8 mg    llm

atorvastatin kept the rules row (with its dose); ramelteon came only from the LLM.


In [5]:
# Build the real hybrid
if ex_rules is not None and ex_llm is not None:
    rules_structured = ex_rules[ex_rules['section'].isin(STRUCTURED_SECTIONS)]
    llm_ids = set(ex_llm['note_id'])
    llm_narrative = ex_llm[ex_llm['note_id'].isin(llm_ids)]

    # keep the comparison fair: only notes both components processed
    common = set(rules_structured['note_id']) | llm_ids
    hybrid = merge_predictions(
        rules_structured[rules_structured['note_id'].isin(common)],
        llm_narrative[llm_narrative['note_id'].isin(common)],
    )
    hybrid.to_parquet(WORK + 'extractions_hybrid.parquet')
    print(f'hybrid: {len(hybrid)} extractions from {hybrid["note_id"].nunique()} notes')
    print(hybrid['source'].value_counts().to_string())
else:
    hybrid = None
    print('Need extractions_rules.parquet and extractions_llm.parquet.')

hybrid: 920 extractions from 177 notes
source
rules    691
llm      229


---
# Part 3 — Final evaluation

All four systems, same gold set, same harness.

In [6]:
gold_path = GOLD + 'gold_v1.csv'

if os.path.exists(gold_path) and hybrid is not None:
    gold = pd.read_csv(gold_path)
    gold_ids = set(gold['note_id'])

    systems = {'rules': ex_rules, 'transformer': ex_ner, 'llm': ex_llm, 'hybrid': hybrid}
    rows = {}
    for name, df in systems.items():
        if df is None:
            continue
        p = df[df['note_id'].isin(gold_ids)].reset_index(drop=True)
        r = evaluate(p, gold)['levels']
        rows[name] = {
            'drug_P': r['drug']['precision'], 'drug_R': r['drug']['recall'],
            'drug_F1': r['drug']['f1'],
            'status_F1': r['drug+status']['f1'],
            'n_pred': len(p),
        }
    final = pd.DataFrame(rows).T
    print(final.to_string())
    final.to_csv(WORK + 'final_comparison.csv')
else:
    print('Need gold_v1.csv and the hybrid. Everything above runs without them.')

             drug_P  drug_R  drug_F1  status_F1  n_pred
rules         0.735   0.598    0.660      0.392   215.0
transformer   0.124   0.121    0.122      0.069   259.0
llm           0.310   0.318    0.314      0.284   271.0
hybrid        0.402   0.481    0.438      0.403   316.0


In [7]:
# Cost/quality table — the other half of the decision
if os.path.exists(gold_path) and hybrid is not None:
    meta_path = WORK + 'llm_run_meta.csv'
    llm_sec = None
    if os.path.exists(meta_path):
        llm_sec = 4.0            # replace with your measured seconds/note from notebook 06
    cost = pd.DataFrame([
        {'system':'rules',       'sec_per_note': 0.01, 'hallucination':'0% (structural)',
         'auditable':'yes', 'gpu':'no'},
        {'system':'transformer', 'sec_per_note': 0.15, 'hallucination':'0% (spans only)',
         'auditable':'partial', 'gpu':'yes'},
        {'system':'llm',         'sec_per_note': llm_sec or 4.0, 'hallucination':'measured, see nb 06',
         'auditable':'no', 'gpu':'yes'},
        {'system':'hybrid',      'sec_per_note': round((llm_sec or 4.0) * float(np.median(share)), 2),
         'hallucination':'narrative lane only', 'auditable':'partial', 'gpu':'yes'},
    ])
    print(cost.to_string(index=False))

     system  sec_per_note       hallucination auditable gpu
      rules          0.01     0% (structural)       yes  no
transformer          0.15     0% (spans only)   partial yes
        llm          4.00 measured, see nb 06        no yes
     hybrid          3.98 narrative lane only   partial yes


### How to read the two tables together

**A higher F1 does not automatically win.** The decision is a tradeoff across four axes:

- **Quality** — F1 at both strictness levels
- **Cost** — seconds per note, GPU requirement
- **Risk** — hallucination exposure, and whether it's structural or measured
- **Auditability** — can a privacy officer or clinician follow why an output appeared?

If the hybrid lands within a point or two of the pure LLM at a fraction of the cost, with hallucination exposure confined to the narrative lane and the medication list itself produced by auditable rules — **that's the system to ship**, even if the LLM's headline F1 is marginally higher.

Writing that reasoning down, with the numbers behind it, is the ADR. It's also the answer to the interview question *"walk me through a technical decision you made and why."*

In [8]:
# Where does each source contribute? -> is the routing actually earning its cost?
if os.path.exists(gold_path) and hybrid is not None:
    hp = hybrid[hybrid['note_id'].isin(gold_ids)].reset_index(drop=True)
    tp, fp, fn = match(hp, gold, 'drug')
    tp_src = Counter(hp.loc[j, 'source'] for j, _ in tp)
    fp_src = Counter(hp.loc[j, 'source'] for j in fp)
    print('true positives by source :', dict(tp_src))
    print('false positives by source:', dict(fp_src))
    print()
    for s in ['rules', 'llm']:
        t, f = tp_src.get(s, 0), fp_src.get(s, 0)
        if t + f:
            print(f'  {s:6} precision within hybrid: {t/(t+f):.3f}  (n={t+f})')
    print()
    print('If the LLM lane contributes few TPs and many FPs, the routing is not paying —')
    print('narrow the narrative lane or raise its threshold.')

true positives by source : {'rules': 82, 'llm': 45}
false positives by source: {'rules': 5, 'llm': 184}

  rules  precision within hybrid: 0.943  (n=87)
  llm    precision within hybrid: 0.197  (n=229)

If the LLM lane contributes few TPs and many FPs, the routing is not paying —
narrow the narrative lane or raise its threshold.


---
# Part 4 — The deliverables

Three documents. **These are what a hiring panel remembers**, and each takes under an hour now that the analysis exists.

In [9]:
adr = f"""# ADR-001: Hybrid rules + LLM architecture for medication extraction

**Status:** Accepted
**Date:** {time.strftime('%Y-%m-%d')}

## Context
Medication extraction from clinical notes, developed on MTSamples (373 notes),
evaluated against a 75-note hand-annotated gold set. Target environment is a
PHIPA-governed hospital: no external LLM APIs, outputs must be auditable.

Three detection approaches were built and measured against the same harness:
rules (lexicon + regex), transformer NER (d4data/biomedical-ner-all), and a
local instruct LLM (Qwen2.5-7B, 4-bit).

## Decision
Route by section type. Structured sections (medications, discharge medications,
allergies) are processed by the rules extractor; narrative sections (HPI,
hospital course, plan) by the LLM. Predictions are merged with rules winning
ties. RAG over RxNorm normalizes surface forms, abstaining below a similarity
threshold.

## Rationale
- Rules are precise, millisecond-fast, and cannot hallucinate by construction.
  Medication lists are the highest-stakes output and rules solve them.
- The LLM's measured advantage is confined to narrative prose: negation scope,
  temporal context, and multi-drug attribution.
- Routing sends roughly {float(np.median(share)):.0%} of text to the expensive lane,
  cutting LLM cost proportionally versus an all-LLM design.
- Every prediction carries a `source` field, so any output is attributable.

## Consequences
Positive: hallucination exposure confined to the narrative lane; the medication
list itself is produced by auditable code; cost scales with narrative volume.

Negative: two components to maintain; routing depends on the sectionizer, so
sectionizer failures propagate; headerless notes fall entirely to the LLM lane.

## Alternatives considered
- **All-rules:** rejected — lexicon-bound recall, crude negation scope.
- **All-LLM:** rejected — cost, hallucination exposure on the medication list,
  and no auditable path for the highest-stakes output.
- **Transformer-only:** rejected — no normalization or status; strictly a
  detection component, better used as a lexicon-gap detector.
- **Fine-tuning:** deferred — 373 notes is too small; revisit with MIMIC access
  and only if hybrid recall proves insufficient.

## Revisit if
- MIMIC evaluation shows the sectionizer failing on real EHR formatting
- Narrative-lane precision within the hybrid falls below the rules lane
- A clinical use case requires sub-second latency end to end
"""

with open(DOCS + 'ADR-001-hybrid-architecture.md', 'w') as f:
    f.write(adr)
print(adr[:900])

# ADR-001: Hybrid rules + LLM architecture for medication extraction

**Status:** Accepted
**Date:** 2026-09-03

## Context
Medication extraction from clinical notes, developed on MTSamples (373 notes),
evaluated against a 75-note hand-annotated gold set. Target environment is a
PHIPA-governed hospital: no external LLM APIs, outputs must be auditable.

Three detection approaches were built and measured against the same harness:
rules (lexicon + regex), transformer NER (d4data/biomedical-ner-all), and a
local instruct LLM (Qwen2.5-7B, 4-bit).

## Decision
Route by section type. Structured sections (medications, discharge medications,
allergies) are processed by the rules extractor; narrative sections (HPI,
hospital course, plan) by the LLM. Predictions are merged with rules winning
ties. RAG over RxNorm normalizes surface forms, abstaining below a similarity
threshold.

## Rationale
- Rul


In [10]:
card = f"""# Model Card: Clinical Medication Extraction

**Version:** 1.0  |  **Date:** {time.strftime('%Y-%m-%d')}

## Intended use
Extraction of medication events (drug, dose, frequency, clinical status) from
free-text clinical notes, for research and development purposes.

**In scope:** methods development, benchmarking, retrospective research cohorts.

**Out of scope — not validated for these:** clinical decision support, medication
reconciliation without human review, any use where output is acted on without a
clinician verifying it against the source note.

## Data
- Developed on MTSamples: 373 transcription samples, publicly available.
- These are teaching transcriptions, NOT EHR exports. They lack copy-forward
  text, EHR templating artifacts, and embedded lab tables.
- Evaluation: 75-note gold set, single annotator, guidelines v1 frozen before
  annotation. Intra-annotator agreement measured on 15 notes.
- No PHI was used at any stage. Public demo runs on open data only.

## Performance
See `final_comparison.csv`. Reported at two strictness levels (drug, drug+status)
with precision and recall separated. Metrics carry the ceiling imposed by
single-annotator agreement — see the gold set documentation.

## Known limitations and failure modes
1. **Negation scope** — simplified NegEx in the rules lane; cues are segment-scoped,
   not syntactically scoped. Known false positives (e.g. "switched to X without
   difficulty").
2. **Lexicon coverage** — drugs absent from RxNorm or below the similarity
   threshold return `needs_review` rather than a mapping.
3. **Formulation detail** — extended-release and combination formulations may
   normalize to the base ingredient.
4. **Hallucination** — the narrative lane uses a generative model. A faithfulness
   check flags extracted drugs absent from the source, but the check is itself
   imperfect (prefix matching tolerates near-misses).
5. **Domain shift** — performance on real EHR notes is unvalidated. MTSamples is
   cleaner and shorter than MIMIC discharge summaries.
6. **Section dependence** — routing relies on the sectionizer; headerless notes
   (~2% of the corpus) route entirely to the narrative lane.

## Privacy and governance
- All inference runs locally. No note content is sent to external APIs.
- This is an architectural constraint, not a configuration option: PHIPA and
  MIMIC's DUA both prohibit third-party transmission of clinical text.
- Quantized (4-bit) local models are used; quantization has a measurable quality
  cost and is a deployment tradeoff, not a free optimization.
- The public demo contains no restricted data and is fed only open-licensed
  or synthetic notes.

## Human oversight
Outputs are decision support at most. Low-confidence normalizations are flagged
`needs_review`; multi-drug segments withhold attributes rather than guess. The
system is designed to abstain rather than assert when uncertain.

## Contact
Maintainer: (your name). Issues: (repo URL).
"""

with open(DOCS + 'MODEL_CARD.md', 'w') as f:
    f.write(card)
print('Wrote docs/MODEL_CARD.md')

Wrote docs/MODEL_CARD.md


In [11]:
roadmap = f"""# Roadmap and deliberate exclusions
_Last updated {time.strftime('%Y-%m-%d')}_

## Deferred, with the evidence that would change the decision

| Deferred | Why | Revisit when |
|---|---|---|
| Fine-tuning a clinical NER model | 373 notes; too small to train on, and pretrained + rules already covers the easy majority | MIMIC access gives 10k+ notes AND hybrid recall proves insufficient |
| Full NegEx/ConText | Simplified version affects ~3.5% of rules output; LLM lane handles the hard cases | Narrative-lane routing is removed, or negation errors exceed 5% |
| Span-level gold annotations | Event-level matching fits the task; span gold would cost another annotation pass | A clean detection-only comparison becomes necessary |
| Vector database (Chroma/FAISS) | ~10^5 concepts fit in a NumPy matrix; a DB adds ops burden for no gain | Vocabulary exceeds ~10^6 vectors or needs persistence + filtering |
| FHIR-formatted output | No consumer for it yet | An integration target exists |
| Multi-annotator gold set | Single annotator is the documented limitation; intra-annotator agreement bounds the metrics | The project moves toward anything clinical-facing |
| Kubernetes / autoscaling | Azure Container Apps handles this scale | Sustained load requires horizontal scaling |

## Next
1. Deploy: Gradio Space (demo), then FastAPI + Docker + Azure Container Apps
2. CI: lint, tests, eval-on-synthetic gate
3. MIMIC transfer: re-annotate 50 notes, rerun the comparison, write the
   generalization-gap analysis
"""

with open(DOCS + 'ROADMAP.md', 'w') as f:
    f.write(roadmap)
print('Wrote docs/ROADMAP.md')
print()
print('docs/ now contains:', os.listdir(DOCS))

Wrote docs/ROADMAP.md

docs/ now contains: ['ROADMAP.md', 'ADR-001-hybrid-architecture.md', 'MODEL_CARD.md']


---
## What you have now

A four-way evaluated system with a documented architecture decision, a model card written for a hospital privacy office, and a roadmap that says what you deliberately didn't build and why.

**The four transferable ideas:**
1. **Route, don't choose.** Components that fail on different inputs should be combined by input type, not ranked.
2. **The highest-stakes output should come from the most auditable component.** Rules produce the medication list; the generative model works on prose.
3. **Provenance is a requirement, not a nicety.** `source` on every row is what makes the system explainable and its components measurable.
4. **The cut-list is a deliverable.** What you declined to build, with the evidence threshold that would change your mind, is the clearest evidence of engineering judgment in the whole project.

**Next — deployment (Phase 7):**
1. **Gradio on Hugging Face Spaces**, one weekend — rules + RAG path only, no GPU, demo notes baked in. Public URL for LinkedIn.
2. **FastAPI → Docker → Azure Container Apps** — the resume line, and post-AZ-900 the quickstart is directly relevant.
3. **GitHub Actions** — lint, pytest, and an eval gate on synthetic notes.
4. **Monitoring** — log latency, extraction counts, confidence distributions; one page that plots them.

Every public deployment runs on open data only — which is itself a governance point for the model card, not a limitation to apologize for.